# Boosting Algorithms — Match Win/Loss Prediction (Gaming Domain)

## 1. Setup

In [ ]:
!pip install xgboost -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

np.random.seed(42)
plt.rcParams["figure.dpi"] = 100


## 2. Dummy Dataset — Match Win/Loss Prediction

Predicting whether a player **wins or loses a match** based on their in-match performance
stats (kills, deaths, assists, damage dealt, etc.) — a natural fit for boosting, since winning
usually depends on *combinations* of stats rather than any single one (e.g. high kills only
matters if deaths are also low).

In [ ]:
n = 3000

kills          = np.random.poisson(6, n)
deaths         = np.random.poisson(5, n)
assists        = np.random.poisson(4, n)
damage_dealt   = np.random.gamma(5, 300, n)
gold_earned    = np.random.gamma(6, 400, n)
healing_done   = np.random.gamma(2, 150, n)
time_survived  = np.random.uniform(5, 30, n)
headshots      = np.random.poisson(2, n)

# a few irrelevant/noisy stats -- real datasets always have some
ping           = np.random.uniform(10, 150, n)
squad_size     = np.random.randint(1, 5, n)
map_id         = np.random.randint(1, 6, n)

# win depends on COMBINATIONS of stats, not single stats in isolation
aggressive_playstyle = ((kills - deaths) > 1) & (damage_dealt > 1400)
support_playstyle    = (assists > 4) & (healing_done > 250) & (deaths < 7)
clutch_playstyle     = (kills > 9) & (time_survived > 20)

signal = (3.2*aggressive_playstyle + 2.8*support_playstyle + 2.5*clutch_playstyle
          + 0.006*gold_earned/100)

z = signal + np.random.randn(n)*0.10 - 2.3
prob_win = 1 / (1 + np.exp(-z))
win = (prob_win > np.random.uniform(0, 1, n)).astype(int)

df = pd.DataFrame({
    "Kills": kills, "Deaths": deaths, "Assists": assists, "Damage_Dealt": damage_dealt,
    "Gold_Earned": gold_earned, "Healing_Done": healing_done, "Time_Survived_Min": time_survived,
    "Headshots": headshots, "Ping": ping, "Squad_Size": squad_size, "Map_ID": map_id,
    "Win": win
})

print("Win rate:", round(df["Win"].mean(), 3))
df.head()


## 3. Train / Test Split

In [ ]:
X = df.drop(columns=["Win"])
y = df["Win"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape


## 4. AdaBoost

In [ ]:
ada_model = AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=42)
ada_model.fit(X_train, y_train)

ada_pred = ada_model.predict(X_test)
ada_prob = ada_model.predict_proba(X_test)[:, 1]

ada_metrics = {
    "Accuracy": accuracy_score(y_test, ada_pred),
    "Precision": precision_score(y_test, ada_pred),
    "Recall": recall_score(y_test, ada_pred),
    "F1": f1_score(y_test, ada_pred),
    "ROC-AUC": roc_auc_score(y_test, ada_prob),
}
ada_metrics


## 5. Gradient Boosting

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)
gb_prob = gb_model.predict_proba(X_test)[:, 1]

gb_metrics = {
    "Accuracy": accuracy_score(y_test, gb_pred),
    "Precision": precision_score(y_test, gb_pred),
    "Recall": recall_score(y_test, gb_pred),
    "F1": f1_score(y_test, gb_pred),
    "ROC-AUC": roc_auc_score(y_test, gb_prob),
}
gb_metrics


## 6. XGBoost

In [ ]:
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                           eval_metric="logloss", random_state=42)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

xgb_metrics = {
    "Accuracy": accuracy_score(y_test, xgb_pred),
    "Precision": precision_score(y_test, xgb_pred),
    "Recall": recall_score(y_test, xgb_pred),
    "F1": f1_score(y_test, xgb_pred),
    "ROC-AUC": roc_auc_score(y_test, xgb_prob),
}
xgb_metrics


## 7. Comparison Table

In [ ]:
comparison_df = pd.DataFrame({
    "AdaBoost": ada_metrics,
    "Gradient Boosting": gb_metrics,
    "XGBoost": xgb_metrics
}).T.round(4)

comparison_df


## 8. Visual Illustration — Metric Comparison

In [ ]:
comparison_df.plot(kind="bar", figsize=(11, 6), colormap="viridis")
plt.title("Boosting Algorithms — Metric Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()


## 9. Visual Illustration — ROC Curves

In [ ]:
plt.figure(figsize=(7, 6))

for name, prob in [("AdaBoost", ada_prob), ("Gradient Boosting", gb_prob), ("XGBoost", xgb_prob)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Visual Illustration — Boosting in Action (Accuracy vs. Number of Trees)

In [ ]:
ada_stage_acc = [accuracy_score(y_test, pred) for pred in ada_model.staged_predict(X_test)]
gb_stage_acc = [accuracy_score(y_test, pred) for pred in gb_model.staged_predict(X_test)]

xgb_stage_acc = []
for i in range(1, 101):
    partial_pred = (xgb_model.predict_proba(X_test, iteration_range=(0, i))[:, 1] > 0.5).astype(int)
    xgb_stage_acc.append(accuracy_score(y_test, partial_pred))

plt.figure(figsize=(9, 6))
plt.plot(range(1, 101), ada_stage_acc, label="AdaBoost", linewidth=2)
plt.plot(range(1, 101), gb_stage_acc, label="Gradient Boosting", linewidth=2)
plt.plot(range(1, 101), xgb_stage_acc, label="XGBoost", linewidth=2)
plt.xlabel("Number of Trees (Boosting Rounds)")
plt.ylabel("Test Accuracy")
plt.title("How Accuracy Improves as More Trees Are Added")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Visual Illustration — Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, [("AdaBoost", ada_model), ("Gradient Boosting", gb_model), ("XGBoost", xgb_model)]):
    importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()
    ax.barh(importances.index, importances.values)
    ax.set_title(name)
    ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()


## 12. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (name, pred) in zip(axes, [("AdaBoost", ada_pred), ("Gradient Boosting", gb_pred), ("XGBoost", xgb_pred)]):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["Loss", "Win"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()


## 13. Summary — Which One Wins?

| Algorithm | Core Idea | Typical Result Here |
|---|---|---|
| AdaBoost | Reweights misclassified samples; uses shallow "stump" trees by default | Lowest accuracy/AUC |
| Gradient Boosting | Each tree fits the residual errors of the previous one; deeper trees | Higher accuracy/AUC |
| XGBoost | Regularized, optimized Gradient Boosting | Highest accuracy/AUC |

**Why the gap is visible here:** winning a match depends on *combinations* of stats — e.g.
high kills only helps if deaths are also low. AdaBoost's default trees are just 1-level
"stumps" (one split, one feature at a time), so they cannot capture these multi-feature
combinations. Gradient Boosting and XGBoost use deeper trees (multiple splits per tree),
letting them learn these interaction patterns directly — which is exactly why they score
noticeably higher on this dataset.

**Ranking (best to weakest, typically):** XGBoost ≥ Gradient Boosting > AdaBoost.